# Field validation — `icearea` (SURF pipeline)

End-to-end validation of every calculated field in the **icearea** surface subset: the notebook RUNs the pipeline, LOADs its own output, and validates each field with dependency-chain maps, PDFs, and a literature comparison.

| | |
|---|---|
| Subset | `icearea` (SURF) |
| Timestep | 2012-11-09 12:00:00 (`20121109_120000`) |
| run_id | `field_validation_v1` |
| Plan | `prompts/field_validation.md` |
| Field reference | `docs/Fields.md` |

Minimal notebook (one raw channel; plan Clarification 6: standalone).  SIarea comes from the LLC_SURF store at this date (transferred; outside the OSN llc_wind window).

## Section 1 — RUN the SURF pipeline for this subset + timestep

One cell (pattern from
`notebooks/notebooks_global/running_generate_global_script.ipynb`).
Existing stores for this date are skipped unless `--clobber` is added,
so re-running is a cheap no-op.

Note: SIarea at 2012-11-09 comes from the LLC_SURF S3 store (OSN llc_wind ends 2012-07-15); the 20121109T12 raw data is transferred and available.

In [ ]:
# Section 1: run the SURF pipeline for this subset and timestep.
SUBSET   = "icearea"
PIPELINE = "SURF"
RUN_ID   = "field_validation_v1"
DATE     = "2012-11-09 12:00:00"   # single validation timestep

!generate-global \
    --config ../../../configs/global/run/field_validation_surface.yaml \
    --pipeline $PIPELINE \
    --subset $SUBSET \
    --run_id $RUN_ID

## Section 2 — LOAD the data generated in Section 1

One cell (pattern from
`notebooks/notebooks_global/assess_generate_global_script.ipynb`):
product reader + grid reader, with shape/date confirmation printout.

In [ ]:
# Section 2: load the store written in Section 1 + the shared grid.
import numpy as np
import matplotlib.pyplot as plt

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid
from dbof.global_dataset_creation.subset_definitions import (
    get_subset_definition,
)

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
BUCKET      = "dbof"
FOLDER      = "surface_fields"       # SURF output folder
DATE_PREFIX = "20121109_120000"      # matches DATE in Section 1

defn = get_subset_definition(PIPELINE, SUBSET)
# get_subset_definition already folds per-pipeline extras (e.g.
# oceQnet for SURF) into model_data_feature_channels.
CHANNELS = (list(defn["model_data_feature_channels"])
            + list(defn["compute_features_channels"]))

fs, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
    dataset_name=defn["dataset_name"], date_prefix=DATE_PREFIX, fs=fs,
)

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=BUCKET, folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat

print(reader)
print(f"channels  : {reader.channel_names}")
print(f"shape     : {reader.shape}  (C, H, W)")
print(f"iteration : {reader.iteration}")
print(f"grid      : XC {XC.shape}, "
      f"lon [{XC.min():.1f}, {XC.max():.1f}], "
      f"lat [{YC.min():.1f}, {YC.max():.1f}]")

## Section 3 — SUBSET: `icearea`

Channels (verbatim from `subset_definitions.SURFACE_SUBSETS`):

raw — `SIarea` (no computed channels)

In [ ]:
# Section 3: guard — store channels must match the code definition.
print(f"subset '{SUBSET}': {len(CHANNELS)} channels")
for ch in CHANNELS:
    print(f"  - {ch}")
assert set(reader.channel_names) == set(CHANNELS), (
    "store channels differ from subset_definitions — regenerate the "
    "store (Section 1, --clobber) or check the code definition")
print("OK: store channel set matches subset_definitions")

## Section 4 — Field & dependency table

| FIELD NAME | UNITS | EQUATION | DEPENDENCIES | LOCATION OF CALC IN CODE |
|---|---|---|---|---|
| SIarea | 0–1 | model output (sea-ice area fraction) | — | LLC_SURF store |

Processing operations: land masking; face→lat-lon stitching; global
downsampling.  No computation, interpolation, rotation, or
differentiation.  Note: mid/low-latitude validation domains are
ice-free — the PDF is a delta at 0 there; the global and SO Atlantic
rows carry the signal (Nov = austral spring: Antarctic ice edge
within the SO Atlantic box).

In [ ]:
# Slice the STORE channels to the validation domains (live fields,
# if any, were sliced in the previous cell) via the shared plumbing
# in dbof.plotting.live_fields.
from dbof.plotting import regions
from dbof.plotting.field_cmaps import load_field_cmaps
from dbof.plotting.live_fields import (
    slice_store, print_region_summary,
)

CMAP_CFG, DIVERGING = load_field_cmaps()

# region_arrays[field][region] = (x, y, arr)
region_arrays = globals().get("region_arrays", {})
SLICE_REGIONS = (regions.REGION_ORDER
                 + globals().get("EXTRA_REGIONS", []))
region_arrays.update(
    slice_store(reader, CHANNELS, XC, YC, SLICE_REGIONS))
print_region_summary(region_arrays)

## Section 5 — Per-field validation

- **Figure 1 — maps**: columns = the field's full dependency chain
  (raw → components → final; every computed step is shown), rows =
  validation domains.  One shared colour scale per column; land/halo
  NaNs gray; regional boxes on the global row.
- **Figure 2 — PDFs**: same grid.  Probability density; land +
  halo-rim NaNs removed; bins shared per field across domains;
  Eq. Pacific row |lat|>2° filtered for f-normalised fields.
- **Literature comparisons** live in Section 6 at the end of the
  notebook — one subsection PER REFERENCE (a reference may validate
  several fields at once), only where a reference exists.  Images in
  `../literature_figures/`, named
  `{field(s)}_{Citation}_{description}.png`.

In [ ]:
# Section 5 helpers: one call per figure, shared by all fields.
from pathlib import Path

import cartopy.crs as ccrs

from dbof.plotting.global_maps import plot_global_field
from dbof.plotting.pipeline_grids import (
    pipeline_map_grid, mask_wrap_cells, LAND_COLOR,
)
from dbof.plotting.pdfs import pipeline_pdf_grid
from dbof.plotting.literature_comparison import side_by_side

# Flat literature directory; files named
# {field}_{Citation}_{description}.png
LIT_DIR = Path("../literature_figures")

# Full dependency chain per field (columns of Figures 1-2), including
# component-level intermediates (gradient / Jacobian components).
CHAINS = {
    "SIarea": ["SIarea"],
}

# f-normalised fields: |lat|>2 deg filter on the Eq. Pacific row of
# the PDFs only (maps annotated instead) — plan Clarification 8.
F_NORM = set()

# Fields drawn/binned on log scales (∝-squared fields).
LOG_FIELDS = set()

PDF_NOTE = ("PDFs: density; land+rim NaNs removed; shared bins "
            "across domains; log10-x for \u221d-squared fields")


# Shared figure toolkit (extracted to dbof.plotting.validation_figures
# — one implementation for all six notebooks); methods bound to the
# historical cell-level names used by the Section 5/6 cells below.
from dbof.plotting.validation_figures import ValidationFigures

_figs = ValidationFigures(
    chains=CHAINS, region_arrays=region_arrays,
    cmap_cfg=CMAP_CFG, diverging=DIVERGING,
    log_fields=LOG_FIELDS, f_norm=F_NORM, lit_dir=LIT_DIR,
)
figure1_maps = _figs.maps
figure2_pdfs = _figs.pdfs
figure3_literature = _figs.literature

### 5.1 SIarea (raw)

Sea-ice area fraction, validated as loaded.  November: Antarctic ice edge in the SO Atlantic box; Arctic cap on the global row.  Open-ocean zeros dominate the PDFs (documented).

In [ ]:
figure1_maps("SIarea")

In [ ]:
figure2_pdfs("SIarea")

## Section 6 — Literature comparisons

One subsection per reference (a reference may validate several fields); only fields with published counterparts appear here.

_No literature references supplied yet for this subset — add per-reference subsections here as they become available._

## Summary — guard-style checks

Pass/fail checklist mirroring `dev/verify_subsets_real_data.py`:
finite fraction, plausible range, and channel-uniqueness guard on the
Gulf Stream domain.

In [ ]:
# Summary: quick stats + uniqueness guard (Gulf Stream domain).
seen = {}
print(f"{'field':24s} {'finite%':>8s} {'min':>11s} "
      f"{'max':>11s} {'unique':>7s}")
for ch in CHANNELS:
    sub = region_arrays[ch]["gulf_stream"][2]
    finite = np.isfinite(sub)
    frac = 100.0 * finite.mean()
    key = hash(sub[finite][::997].tobytes()) if finite.any() else ch
    dup = seen.get(key)
    seen[key] = ch
    print(f"{ch:24s} {frac:7.1f}% {np.nanmin(sub):11.3g} "
          f"{np.nanmax(sub):11.3g} {'DUP!' if dup else 'ok':>7s}")
    assert dup is None, f"{ch} identical to {dup}"
print("\nAll checks passed.")

**Cross-references** — none; DEPTH `icearea` is surface_only and references this notebook.